 Data Quality Checks**

Goal:
- Load a real world dataset
- Run common data quality checks
- Build a simple data quality checklist
---

**1. Setup**

In [2]:
%pip install -q pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50) # Show all columns in the DataFrame 

**2. Load the dataset**

In [55]:
df = pd.read_csv("titanic_dataset.csv")

In [42]:
print(df.shape)

(891, 15)


In [43]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


**3. Adding few columns for demonstrating data issues**

In [44]:
np.arange(1,10)

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [71]:


df["passenger_id"] = np.arange(1, len(df)+1) # Add a new column with unique IDs starting from 1
df["constant_col"] = 1 # Add a new column with a constant value


In [45]:
# create a dirty version of embarked_town
df["embark_town_dirty"] = df["embark_town"].copy()

In [46]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,embark_town_dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,Southampton


In [47]:
# select a few randowm rows(only non-missing)
sample_idx = df["embark_town_dirty"].dropna().sample(6,random_state=42).index

In [48]:
sample_idx

Index([281, 435, 39, 418, 585, 804], dtype='int64')

In [49]:

#apply simple dirty transformation
df.loc[sample_idx[0:2], "embark_town_dirty"] = df.loc[sample_idx[0:2], "embark_town_dirty"].str.upper() # make uppercase
df.loc[sample_idx[2:4], "embark_town_dirty"] = df.loc[sample_idx[2:4], "embark_town_dirty"].str.strip().apply(lambda x: f" {x} ") # remove last character
df.loc[sample_idx[4:6], "embark_town_dirty"] = "unknown" # replace with unknown

In [50]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,embark_town_dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,Southampton


In [54]:
df.iloc[435]["embark_town_dirty"] # check the dirty value
df.iloc[39]["embark_town_dirty"] # check the dirty value
df.iloc[804]["embark_town_dirty"] # check the dirty value

'unknown'

**4. Data quality checklist**

Quick checklist for this video:
1. Basic dataset overview
2. Missing values summary
3. Duplicates
4. Data type validation
5. Constant and quasi constant columns
6. ID like columns
7. String inconsistencies
8. High null columns
9. High zero columns (for numeric features)

Basic Dataset Overview

In [56]:
df.shape

(891, 15)

In [57]:
df.dtypes

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object

In [58]:
print(df.shape)
print("-"*50)
print(df.dtypes)
print("-"*50)
print(df.info())

(891, 15)
--------------------------------------------------
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class     

In [59]:
df.nunique()

survived         2
pclass           3
sex              2
age             88
sibsp            7
parch            7
fare           248
embarked         3
class            3
who              3
adult_male       2
deck             7
embark_town      3
alive            2
alone            2
dtype: int64

**2. Missing values summary**

In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


In [60]:
missing_count = df.isna().sum().sort_values(ascending=False)
print(missing_count)

deck           688
age            177
embarked         2
embark_town      2
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
class            0
who              0
adult_male       0
alive            0
alone            0
dtype: int64


In [62]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }
)
print("Missing Values Summary")
missing_summary

Missing Values Summary


,missing_count,missing_percent
deck,688,77.216611
age,177,19.865320
embarked,2,0.224467
embark_town,2,0.224467
survived,0,0.000000
pclass,0,0.000000
sex,0,0.000000
sibsp,0,0.000000
parch,0,0.000000
fare,0,0.000000


**3.Duplicates**

In [63]:
duplicates_mask = df.duplicated()
num_duplicates = duplicates_mask.sum()

print("Number of duplicate rows:", num_duplicates)

Number of duplicate rows: 107


**4. Data Type Validation**

In [65]:
expected_types = {
    "survived": "int64",
    "pclass": "int64",
    "sex": "category",
    "age": "float64",
    "fare": "float64",
    "embark_town": "category",
    "passenger_id": "int64"
}

for col, expected_type in expected_types.items():
    print(col,expected_type)

survived int64
pclass int64
sex category
age float64
fare float64
embark_town category
passenger_id int64


In [64]:
expected_types = {
    "survived": "int64",
    "pclass": "int64",
    "sex": "category",
    "age": "float64",
    "fare": "float64",
    "embark_town": "category",
    "passenger_id": "int64"
}

print("Data Type Validation:")
for col, expected in expected_types.items():
    if col in df.columns:
        actual=df[col].dtype
        print(f"{col}: actual={actual}, expected={expected}")

Data Type Validation:
survived: actual=int64, expected=int64
pclass: actual=int64, expected=int64
sex: actual=str, expected=category
age: actual=float64, expected=float64
fare: actual=float64, expected=float64
embark_town: actual=str, expected=category


In [68]:
# example of fixing a data type if needed
df["pclass"] = df["pclass"].astype("int32")
df["pclass"].dtype
df["pclass"] = df["pclass"].astype("int64")
df["pclass"].dtype

dtype('int64')

**5. Constant & Quasi-Constant columns**

In [72]:
n_rows =len(df)
nunique = df.nunique()


constant_cols = nunique[nunique == 1].index.tolist() # Get the list of columns that have only one unique value (constant columns)

print("Constant columns:", constant_cols)


Constant columns: ['constant_col']


In [73]:
df.nunique()

survived          2
pclass            3
sex               2
age              88
sibsp             7
parch             7
fare            248
embarked          3
class             3
who               3
adult_male        2
deck              7
embark_town       3
alive             2
alone             2
passenger_id    891
constant_col      1
dtype: int64

Quasi-constant feature: a column that has very low variance—one dominant value appears in almost all rows, with only a tiny fraction of other values.

Why it matters: Quasi-constant features have minimal predictive power and waste model capacity.

When to remove:

If dominant value > 95-99% of data, consider dropping it.
It adds noise without signal.


In [74]:
n_rows =len(df)
nunique = df.nunique()


constant_cols = nunique[nunique == 1].index.tolist() # Get the list of columns that have only one unique value (constant columns)

print("Constant columns:", constant_cols)

# quasi constant columns have very low variance and can be considered for removal as they may not provide much information for modeling.
quasi_constant_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).values[0] # Get the frequency of the most common value
    if top_freq > 0.95: # If the most common value accounts for more than 95% of the data
        quasi_constant_cols.append(col)

print("Quasi-constant columns:", quasi_constant_cols)

Constant columns: ['constant_col']
Quasi-constant columns: ['constant_col']


In [75]:
df["survived"].value_counts()

survived
0    549
1    342
Name: count, dtype: int64

In [76]:
df["survived"].value_counts(normalize=True)

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

**4.6. ID like columns**

Columns where number of unique values is closer to number of df rows

In [77]:
n_rows = len(df)

id_like_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == n_rows:
        id_like_cols.append(col)

print("ID like columns:", id_like_cols)

ID like columns: ['passenger_id']


**4.7. String Inconsistencies**

In [ ]:
object_cols = df.select_dtypes(include=["object", "category"]).columns.to_list() # Select columns with object or category data types and convert to a list
print("Object or category based columns", object_cols)



Object or category based columns ['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']


/var/folders/zp/zh_rycfs4j14qq7_xth9cy8h0000gn/T/ipykernel_88063/159297531.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object", "category"]).columns.to_list()


In [80]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,passenger_id,constant_col
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,1
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,1
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3,1
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4,1
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5,1


In [79]:
df.dtypes

survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class               str
who                 str
adult_male         bool
deck                str
embark_town         str
alive               str
alone              bool
passenger_id      int64
constant_col      int64
dtype: object

In [ ]:
# simple clean: strip spaces convert it to lower case
df["embark_town_clean"] = (
    df["embark_town_dirty"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("unknown", np.nan)
) # Create a new column "embark_town_clean" by cleaning the "embark_town_dirty" 
#column: convert to string, strip spaces, convert to lowercase, and replace "unknown" with NaN

**4.8. High null columns**

In [ ]:
high_null_threshold = 0.4   # 40% or more values are missing in a columns

high_null_cols = missing_summary[missing_summary["missing_percent"] >= high_null_threshold * 100] 
# Filter columns where missing percentage is greater than or equal to the threshold

print("Columns with high missing percentage:")
print(high_null_cols)

Columns with high missing percentage:
      missing_count  missing_percent
deck            688        77.216611


**4.9. High zero columns (numeric)**

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.to_list() # Select columns with numeric data types and convert to a list
zero_share = {} # Initialize an empty dictionary to store the share of zeros for each numeric column

for col in numeric_cols: # Iterate over each numeric column
    zero_share[col] = (df[col] == 0).mean() # Calculate the share of zeros in the column and store it in the dictionary

zero_share_series = pd.Series(zero_share).sort_values(ascending=False) # Convert the dictionary to a pandas Series and sort it in descending order
print(zero_share_series)

print("-"*50)

high_zero_threshold = 0.8  # 80 percent or more zeros
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print("Numeric columns with many zeros:", high_zero_cols)

parch           0.760943
sibsp           0.682379
survived        0.616162
fare            0.016835
pclass          0.000000
age             0.000000
passenger_id    0.000000
constant_col    0.000000
dtype: float64
--------------------------------------------------
Numeric columns with many zeros: Series([], dtype: float64)


In [84]:
df_zeros = df.copy() # Create a copy of the original DataFrame to work with zeros

df_zeros["col_zeros"] = 0 # Add a new column "col_zeros" initialized to 0, which will be used to demonstrate the presence of zeros in numeric columns

# numeric_cols = df_zeros.select_dtypes(include=[np.number]).columns.to_list()
numeric_cols = ["age", "constant_col", "fare", "col_zeros"] # Define a list of numeric columns to analyze, including the new "col_zeros" column which contains only zeros
zero_share = {}

for col in numeric_cols:
    zero_share[col] = (df_zeros[col] == 0).mean()

zero_share_series = pd.Series(zero_share).sort_values(ascending=False)
print(zero_share_series)

print("-"*50)

high_zero_threshold = 0.8  # 80 percent or more zeros
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print("Numeric columns with many zeros:", high_zero_cols)

col_zeros       1.000000
fare            0.016835
age             0.000000
constant_col    0.000000
dtype: float64
--------------------------------------------------
Numeric columns with many zeros: col_zeros    1.0
dtype: float64
